In [ ]:
# Wstępna analiza danych - Fraud Detection

# 1. Import niezbędnych bibliotek
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import seaborn as sns
import json
from datetime import datetime

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:

print("Wczytywanie danych o transakcjach...")
transactions_df = pd.read_json('../data/raw/transactions.json', lines=True)
print(f"Wymiary zbioru transakcji: {transactions_df.shape}")

print("\nWczytywanie danych o sprzedawcach...")
merchants_df = pd.read_csv('../data/raw/merchants.csv')
print(f"Wymiary zbioru sprzedawców: {merchants_df.shape}")

print("\nWczytywanie danych o użytkownikach...")
users_df = pd.read_csv('../data/raw/users.csv')
print(f"Wymiary zbioru użytkowników: {users_df.shape}")


🔍 **Syntetyczne Transakcje Kartowe – Wprowadzenie do Wykrywania Oszustw**  

### Witamy w zestawie startowym hackathonu Mastercard!  

Ten syntetyczny zestaw danych symuluje aktywność transakcyjną kart kredytowych, uwzględniając zarówno legalne, jak i oszukańcze zdarzenia. Zaprojektowany tak, aby odzwierciedlać rzeczywiste struktury danych, umożliwia podstawowe eksperymenty z wykrywaniem oszustw.  

---

### 🔢 **Co zawiera zestaw?**  
- **`transactions.json`**: 500 000 transakcji w formacie JSON o złożonej strukturze (lokalizacja, metoda płatności, dane sesji itp.).  
- **`users.csv`**: 20 000 użytkowników z Europy z danymi demograficznymi i finansowymi.  
- **`merchants.csv`**: 1000 europejskich sprzedawców z cechami behawioralnymi i wskaźnikami wiarygodności.  

---

### 🧠 **Proponowane Zadanie Startowe**  
Zbuduj model uczenia maszynowego klasyfikujący transakcje jako:  
- **oszukańcze** (`1`),  
- **legalne** (`0`).  

**Kroki:**  
1. Wytrenuj model na podstawie danych transakcyjnych, użytkowników i sprzedawców.  
2. Przygotuj predykcje dla nowej partii transakcji.  

---

#### 🔎 **Punkty do eksploracji:**  
- **Wzorce czasowe oszustw** (np. godziny/dni z większą liczbą oszustw),  
- **Ryzykowne regiony i sprzedawcy** (identyfikacja podatnych lokalizacji),  
- **Anomalie behawioralne** (np. nietypowe wydatki użytkowników).  

In [ ]:
transactions_df

In [ ]:
transactions_df.columns

In [ ]:
# Lista kolumn do wykluczenia
exclude_columns = {'transaction_id', 'user_id', 'merchant_id'}

# Tworzymy słownik mapujący stare nazwy na nowe (z suffixem "_transaction")
rename_dict = {
    col: f"{col}_transaction" 
    for col in transactions_df.columns 
    if col not in exclude_columns
}

# Zmieniamy nazwy kolumn w DataFrame
transactions_df.rename(columns=rename_dict, inplace=True)

| Przed zmianą                 | Po zmianie                     |
|------------------------------|--------------------------------|
| `transaction_id`             | `transaction_id` *(bez zmian)* |
| `timestamp`                  | `timestamp_transaction`        |
| `user_id`                    | `user_id` *(bez zmian)*        |
| `merchant_id`                | `merchant_id` *(bez zmian)*    |
| `amount`                     | `amount_transaction`           |
| `channel`                    | `channel_transaction`          |
| `currency`                   | `currency_transaction`         |
| `device`                     | `device_transaction`           |
| `location`                   | `location_transaction`         |
| `payment_method`             | `payment_method_transaction`   |
| `is_international`           | `is_international_transaction` |
| `session_length_seconds`     | `session_length_seconds_transaction` |
| `is_first_time_merchant`     | `is_first_time_merchant_transaction` |
| `is_fraud`                   | `is_fraud_transaction`         |

In [ ]:
merchants_df

In [ ]:
merchants_df.columns

In [ ]:
rename_dict = {
    col: f"{col}_merchant"
    for col in merchants_df.columns
    if col != 'merchant_id'
}
merchants_df.rename(columns=rename_dict, inplace=True)

| Przed zmianą                  | Po zmianie                     |
|-------------------------------|--------------------------------|
| `merchant_id`                 | `merchant_id` *(bez zmian)*    |
| `category`                    | `category_merchant`            |
| `country`                     | `country_merchant`             |
| `trust_score`                 | `trust_score_merchant`         |
| `number_of_alerts_last_6_months` | `number_of_alerts_last_6_months_merchant` |
| `avg_transaction_amount`      | `avg_transaction_amount_merchant` |
| `account_age_months`          | `account_age_months_merchant`  |
| `has_fraud_history`           | `has_fraud_history_merchant`   |

In [ ]:
users_df

In [ ]:
users_df.columns

In [ ]:
# Lub Metoda 2: Słownik rename
users_df.rename(columns={
    col: f"{col}_user" 
    for col in users_df.columns 
    if col != 'user_id'
}, inplace=True)

| Przed zmianą                  | Po zmianie                     |
|-------------------------------|--------------------------------|
| `user_id`                     | `user_id` *(bez zmian)*        |
| `age`                         | `age_user`                     |
| `sex`                         | `sex_user`                     |
| `education`                   | `education_user`               |
| `primary_source_of_income`     | `primary_source_of_income_user`|
| `sum_of_monthly_installments`  | `sum_of_monthly_installments_user`|
| `sum_of_monthly_expenses`      | `sum_of_monthly_expenses_user` |
| `country`                     | `country_user`                 |
| `signup_date`                 | `signup_date_user`             |
| `risk_score`                  | `risk_score_user`              |

In [ ]:
# Checking for potential join columns
print("Potential join columns between transactions_df and merchants_df:")
print(set(transactions_df.columns).intersection(merchants_df.columns))

print("\nPotential join columns between transactions_df and users_df:")
print(set(transactions_df.columns).intersection(users_df.columns))

print("\nPotential join columns between merchants_df and users_df:")
print(set(merchants_df.columns).intersection(users_df.columns))

In [ ]:
# Joining transactions_df with merchants_df on 'merchant_id'
merged_df = transactions_df.merge(merchants_df, on='merchant_id', how='left')

# Joining the resulting dataframe with users_df on 'user_id'
merged_df = merged_df.merge(users_df, on='user_id', how='left')

In [ ]:
merged_df

In [ ]:
merged_df.dtypes


In [ ]:
from ydata_profiling import ProfileReport

# Generate the profiling report
profile = ProfileReport(merged_df, title="Merged DataFrame Profiling Report", explorative=True)

# Save the report to an HTML file
profile.to_file("merged_df_profiling_report.html")

# Display the report in the notebook (optional)
profile.to_notebook_iframe()

In [ ]:
# 3. Podstawowa eksploracja danych
print("=== Analiza transakcji ===")
print("\nPrzykładowe transakcje:")
display(transactions_df.head())

print("\nInformacje o typach danych:")
display(transactions_df.info())

print("\nStatystyki opisowe:")
display(transactions_df.describe())

print("\nLiczba transakcji oszukańczych vs legalnych:")
fraud_counts = transactions_df['is_fraud'].value_counts()
print(fraud_counts)
print(f"\nProcent transakcji oszukańczych: {fraud_counts[1]/len(transactions_df)*100:.2f}%")


In [ ]:
# 4. Wizualizacje
# Rozkład transakcji oszukańczych vs legalnych
plt.figure(figsize=(10, 6))
sns.countplot(data=transactions_df, x='is_fraud')
plt.title('Rozkład transakcji oszukańczych vs legalnych')
plt.xlabel('Czy oszustwo')
plt.ylabel('Liczba transakcji')
plt.show()

# Rozkład kwot transakcji
plt.figure(figsize=(12, 6))
sns.histplot(data=transactions_df, x='amount', hue='is_fraud', bins=50)
plt.title('Rozkład kwot transakcji')
plt.xlabel('Kwota transakcji')
plt.ylabel('Liczba transakcji')
plt.show()

# Rozkład kwot transakcji (skala logarytmiczna)
plt.figure(figsize=(12, 6))
sns.histplot(data=transactions_df, x='amount', hue='is_fraud', bins=50)
plt.xscale('log')
plt.title('Rozkład kwot transakcji (skala logarytmiczna)')
plt.xlabel('Kwota transakcji (log)')
plt.ylabel('Liczba transakcji')
plt.show()



In [ ]:
# 5. Analiza korelacji
# Wybór kolumn numerycznych
numeric_columns = transactions_df.select_dtypes(include=[np.number]).columns

# Macierz korelacji
plt.figure(figsize=(12, 8))
correlation_matrix = transactions_df[numeric_columns].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Macierz korelacji')
plt.show()

In [ ]:


# 6. Analiza czasowa
# Konwersja timestamp na datetime
transactions_df['timestamp'] = pd.to_datetime(transactions_df['timestamp'])

# Dodanie kolumn czasowych
transactions_df['hour'] = transactions_df['timestamp'].dt.hour
transactions_df['day_of_week'] = transactions_df['timestamp'].dt.dayofweek

# Rozkład transakcji w ciągu dnia
plt.figure(figsize=(12, 6))
sns.countplot(data=transactions_df, x='hour', hue='is_fraud')
plt.title('Rozkład transakcji w ciągu dnia')
plt.xlabel('Godzina')
plt.ylabel('Liczba transakcji')
plt.show()

# Rozkład transakcji w ciągu tygodnia
plt.figure(figsize=(12, 6))
sns.countplot(data=transactions_df, x='day_of_week', hue='is_fraud')
plt.title('Rozkład transakcji w ciągu tygodnia')
plt.xlabel('Dzień tygodnia')
plt.ylabel('Liczba transakcji')
plt.show()



In [ ]:
# 7. Podsumowanie wstępnej analizy
print("=== Podsumowanie wstępnej analizy ===\n")

# Podstawowe statystyki
print(f"Liczba transakcji: {len(transactions_df)}")
print(f"Liczba sprzedawców: {len(merchants_df)}")
print(f"Liczba użytkowników: {len(users_df)}")
print(f"\nProcent transakcji oszukańczych: {fraud_counts[1]/len(transactions_df)*100:.2f}%")

# Statystyki kwot
print("\nStatystyki kwot transakcji:")
print(transactions_df.groupby('is_fraud')['amount'].describe())

# Informacje o brakujących danych
print("\nLiczba brakujących wartości w każdej kolumnie:")
print(transactions_df.isnull().sum())

In [ ]:
# Analiza liczby transakcji na użytkownika
user_transaction_counts = transactions_df.groupby('user_id').size().reset_index(name='transaction_count')

plt.figure(figsize=(12, 6))
sns.histplot(data=user_transaction_counts, x='transaction_count', bins=50)
plt.title('Rozkład liczby transakcji na użytkownika')
plt.xlabel('Liczba transakcji')
plt.ylabel('Liczba użytkowników')
plt.show()

print("Statystyki liczby transakcji na użytkownika:")
print(user_transaction_counts['transaction_count'].describe())

In [ ]:
# Analiza średniej wartości transakcji na użytkownika
user_avg_amount = transactions_df.groupby('user_id')['amount'].agg(['mean', 'std']).reset_index()

plt.figure(figsize=(12, 6))
sns.scatterplot(data=user_avg_amount, x='mean', y='std')
plt.title('Średnia vs odchylenie standardowe kwot transakcji na użytkownika')
plt.xlabel('Średnia kwota transakcji')
plt.ylabel('Odchylenie standardowe kwot')
plt.show()

print("Statystyki średniej kwoty transakcji na użytkownika:")
print(user_avg_amount['mean'].describe())

In [ ]:
# Analiza częstotliwości transakcji
transactions_sorted = transactions_df.sort_values(['user_id', 'timestamp'])
transactions_sorted['time_diff'] = transactions_sorted.groupby('user_id')['timestamp'].diff()
transactions_sorted['time_diff_minutes'] = transactions_sorted['time_diff'].dt.total_seconds() / 60

plt.figure(figsize=(12, 6))
sns.histplot(data=transactions_sorted[transactions_sorted['time_diff_minutes'] < 60], 
             x='time_diff_minutes', hue='is_fraud', bins=50)
plt.title('Rozkład odstępów czasowych między transakcjami (do 60 minut)')
plt.xlabel('Odstęp czasowy (minuty)')
plt.ylabel('Liczba transakcji')
plt.show()

print("Statystyki odstępów czasowych między transakcjami:")
print(transactions_sorted['time_diff_minutes'].describe())

In [ ]:
# Analiza wzorców płatności
payment_patterns = transactions_df.groupby(['user_id', 'payment_method']).size().unstack(fill_value=0)
payment_patterns['total_transactions'] = payment_patterns.sum(axis=1)
payment_patterns['credit_card_ratio'] = payment_patterns['credit_card'] / payment_patterns['total_transactions']

plt.figure(figsize=(12, 6))
sns.histplot(data=payment_patterns, x='credit_card_ratio', bins=50)
plt.title('Rozkład proporcji transakcji kartą kredytową na użytkownika')
plt.xlabel('Proporcja transakcji kartą kredytową')
plt.ylabel('Liczba użytkowników')
plt.show()

In [ ]:
# Analiza geograficzna
# Rozpakowanie współrzędnych
transactions_df['latitude'] = transactions_df['location'].apply(lambda x: x['lat'])
transactions_df['longitude'] = transactions_df['location'].apply(lambda x: x['long'])

plt.figure(figsize=(12, 8))
sns.scatterplot(data=transactions_df, x='longitude', y='latitude', hue='is_fraud', alpha=0.5)
plt.title('Rozkład geograficzny transakcji')
plt.xlabel('Długość geograficzna')
plt.ylabel('Szerokość geograficzna')
plt.show()

In [ ]:
import plotly.graph_objects as go
import json

# Konwersja kolumny location na osobne kolumny lat i long (ta sama funkcja co wcześniej)
def extract_coordinates(loc):
    if isinstance(loc, str):
        try:
            loc_dict = json.loads(loc)
            return loc_dict['lat'], loc_dict['long']
        except:
            return None, None
    elif isinstance(loc, dict):
        return loc['lat'], loc['long']
    return None, None

# Ekstrakcja współrzędnych
transactions_df['latitude'], transactions_df['longitude'] = zip(*transactions_df['location'].apply(extract_coordinates))

# Usunięcie wierszy z brakującymi współrzędnymi
transactions_df = transactions_df.dropna(subset=['latitude', 'longitude'])

# Przygotowanie danych
fraud = transactions_df[transactions_df['is_fraud'] == 1]
legit = transactions_df[transactions_df['is_fraud'] == 0]

# Tworzenie mapy
fig = go.Figure()

# Warstwa transakcji oszukańczych
fig.add_trace(go.Densitymapbox(
    lat=fraud['latitude'],
    lon=fraud['longitude'],
    radius=15,
    opacity=0.7,
    colorscale='Reds',
    name='Transakcje oszukańcze',
    showlegend=True
))

# Warstwa transakcji legalnych z customową paletą kolorów
fig.add_trace(go.Densitymapbox(
    lat=legit['latitude'],
    lon=legit['longitude'],
    radius=10,
    opacity=0.5,
    colorscale=[[0.0, 'blue'], [0.4, 'blue'], [0.65, 'lime'], [1.0, 'red']],
    name='Transakcje legalne',
    showlegend=True
))

# Konfiguracja mapy
fig.update_layout(
    mapbox_style="carto-positron",
    mapbox_center={
        "lat": transactions_df['latitude'].mean(),
        "lon": transactions_df['longitude'].mean()
    },
    mapbox_zoom=3,
    margin={"r":0,"t":0,"l":0,"b":0},
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()

In [ ]:
# 8. Analiza geograficzna
import folium
from folium.plugins import HeatMap
import json

# Konwersja kolumny location na osobne kolumny lat i long
def extract_coordinates(loc):
    if isinstance(loc, str):
        try:
            loc_dict = json.loads(loc)
            return loc_dict['lat'], loc_dict['long']
        except:
            return None, None
    elif isinstance(loc, dict):
        return loc['lat'], loc['long']
    return None, None

# Ekstrakcja współrzędnych
transactions_df['latitude'], transactions_df['longitude'] = zip(*transactions_df['location'].apply(extract_coordinates))

# Usunięcie wierszy z brakującymi współrzędnymi
transactions_df = transactions_df.dropna(subset=['latitude', 'longitude'])

# Tworzenie mapy
m = folium.Map(location=[transactions_df['latitude'].mean(), transactions_df['longitude'].mean()],
               zoom_start=4)

# Przygotowanie danych do HeatMap
fraud_locations = transactions_df[transactions_df['is_fraud'] == 1][['latitude', 'longitude']].values.tolist()
legit_locations = transactions_df[transactions_df['is_fraud'] == 0][['latitude', 'longitude']].values.tolist()

# Dodanie warstwy z transakcjami oszukańczymi
HeatMap(fraud_locations, radius=15, blur=10, max_zoom=1, name='Transakcje oszukańcze').add_to(m)

# Dodanie warstwy z transakcjami legalnymi
HeatMap(legit_locations, radius=10, blur=5, max_zoom=1, 
        gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}, name='Transakcje legalne').add_to(m)

# Dodanie kontrolki warstw
folium.LayerControl().add_to(m)

# Wyświetlenie mapy
m

In [ ]:
import folium
from folium.plugins import HeatMap
import json

# Funkcja ekstrakcji współrzędnych
def extract_coordinates(loc):
    if isinstance(loc, str):
        try:
            loc_dict = json.loads(loc)
            return loc_dict['lat'], loc_dict['long']
        except:
            return None, None
    elif isinstance(loc, dict):
        return loc['lat'], loc['long']
    return None, None

# Ekstrakcja współrzędnych
transactions_df['latitude'], transactions_df['longitude'] = zip(*transactions_df['location'].apply(extract_coordinates))

# Usunięcie brakujących współrzędnych
transactions_df = transactions_df.dropna(subset=['latitude', 'longitude'])

# Obliczenie środka mapy
map_center = [
    transactions_df['latitude'].mean(),
    transactions_df['longitude'].mean()
]

# Inicjalizacja mapy
m = folium.Map(
    location=map_center,
    zoom_start=4,
    tiles='CartoDB positron',
    attr='CartoDB'  # Atrybucja dla domyślnych płytek
)

# Przygotowanie danych
fraud_locations = transactions_df[transactions_df['is_fraud'] == 1][['latitude', 'longitude']].values.tolist()
legit_locations = transactions_df[transactions_df['is_fraud'] == 0][['latitude', 'longitude']].values.tolist()

# Warstwa oszustw
HeatMap(
    fraud_locations,
    name='Transakcje oszukańcze',
    radius=15,
    blur=10,
    max_zoom=1,
    gradient={0.1: 'yellow', 0.5: 'orange', 1: 'red'},
    show=False
).add_to(m)

# Warstwa legalnych transakcji
HeatMap(
    legit_locations,
    name='Transakcje legalne',
    radius=10,
    blur=5,
    max_zoom=1,
    gradient={'0.4': 'blue', '0.65': 'lime', '1': 'red'},
    show=True
).add_to(m)

# Kontrolka warstw
folium.LayerControl(
    position='topright',
    collapsed=False
).add_to(m)

# Dodatkowe opcje
m.add_child(folium.LatLngPopup())

# Poprawiona warstwa Stamen Toner z atrybucją
folium.TileLayer(
    'Stamen Toner',
    name='Czarno-biała',
    attr='Map tiles by Stamen Design, CC BY 3.0 — Data by OpenStreetMap, ODbL'
).add_to(m)

m

In [ ]:
# 8. Analiza geograficzna
import folium
from folium.plugins import HeatMap
import json

# Konwersja kolumny location na osobne kolumny lat i long
def extract_coordinates(loc):
    if isinstance(loc, str):
        try:
            loc_dict = json.loads(loc)
            return loc_dict['lat'], loc_dict['long']
        except:
            return None, None
    elif isinstance(loc, dict):
        return loc['lat'], loc['long']
    return None, None

# Ekstrakcja współrzędnych
transactions_df['latitude'], transactions_df['longitude'] = zip(*transactions_df['location'].apply(extract_coordinates))

# Usunięcie wierszy z brakującymi współrzędnymi
transactions_df = transactions_df.dropna(subset=['latitude', 'longitude'])

# Tworzenie mapy
m = folium.Map(location=[transactions_df['latitude'].mean(), transactions_df['longitude'].mean()],
               zoom_start=4)

# Przygotowanie danych do HeatMap
fraud_locations = transactions_df[transactions_df['is_fraud'] == 1][['latitude', 'longitude']].values.tolist()
legit_locations = transactions_df[transactions_df['is_fraud'] == 0][['latitude', 'longitude']].values.tolist()

# Dodanie warstwy z transakcjami oszukańczymi
HeatMap(fraud_locations, radius=15, blur=10, max_zoom=1, name='Transakcje oszukańcze').add_to(m)

# Dodanie warstwy z transakcjami legalnymi
HeatMap(legit_locations, radius=10, blur=5, max_zoom=1, 
        gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}, name='Transakcje legalne').add_to(m)

# Dodanie kontrolki warstw
folium.LayerControl().add_to(m)

# Wyświetlenie mapy
m

In [ ]:
# 8. Analiza geograficzna
import folium
from folium.plugins import HeatMap
import json

# Konwersja kolumny location na osobne kolumny lat i long
def extract_coordinates(loc):
    if isinstance(loc, str):
        try:
            loc_dict = json.loads(loc)
            return loc_dict['lat'], loc_dict['long']
        except:
            return None, None
    elif isinstance(loc, dict):
        return loc['lat'], loc['long']
    return None, None

# Ekstrakcja współrzędnych
transactions_df['latitude'], transactions_df['longitude'] = zip(*transactions_df['location'].apply(extract_coordinates))

# Usunięcie wierszy z brakującymi współrzędnymi
transactions_df = transactions_df.dropna(subset=['latitude', 'longitude'])

# Tworzenie mapy
m = folium.Map(location=[transactions_df['latitude'].mean(), transactions_df['longitude'].mean()],
               zoom_start=4)

# Przygotowanie danych do HeatMap
fraud_locations = transactions_df[transactions_df['is_fraud'] == 1][['latitude', 'longitude']].values.tolist()
legit_locations = transactions_df[transactions_df['is_fraud'] == 0][['latitude', 'longitude']].values.tolist()

# Dodanie warstwy z transakcjami oszukańczymi
HeatMap(fraud_locations, radius=15, blur=10, max_zoom=1, name='Transakcje oszukańcze').add_to(m)

# Dodanie warstwy z transakcjami legalnymi
HeatMap(legit_locations, radius=10, blur=5, max_zoom=1, 
        gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'}, name='Transakcje legalne').add_to(m)

# Dodanie kontrolki warstw
folium.LayerControl().add_to(m)

# Wyświetlenie mapy
m

In [ ]:
# Analiza gęstości transakcji w różnych regionach
print("\n=== Analiza gęstości transakcji ===")
print("\nTop 5 regionów z największą liczbą transakcji oszukańczych:")
fraud_by_region = transactions_df[transactions_df['is_fraud'] == 1].groupby(['latitude', 'longitude']).size().reset_index(name='count')
print(fraud_by_region.sort_values('count', ascending=False).head())

print("\nTop 5 regionów z największą liczbą transakcji legalnych:")
legit_by_region = transactions_df[transactions_df['is_fraud'] == 0].groupby(['latitude', 'longitude']).size().reset_index(name='count')
print(legit_by_region.sort_values('count', ascending=False).head())

In [ ]:
# Analiza korelacji między cechami
numeric_features = ['amount', 'session_length_seconds', 'is_international', 'is_first_time_merchant']
correlation_matrix = transactions_df[numeric_features + ['is_fraud']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Macierz korelacji cech numerycznych')
plt.show()

In [ ]:
# Analiza anomalii w zachowaniu użytkowników
user_stats = transactions_df.groupby('user_id').agg({
    'amount': ['mean', 'std', 'max'],
    'session_length_seconds': ['mean', 'std'],
    'is_international': 'mean',
    'is_fraud': 'mean'
}).reset_index()

# Wykrywanie anomalii w kwotach transakcji
user_stats['amount_zscore'] = (user_stats[('amount', 'max')] - user_stats[('amount', 'mean')]) / user_stats[('amount', 'std')]

plt.figure(figsize=(12, 6))
sns.histplot(data=user_stats, x='amount_zscore', bins=50)
plt.title('Rozkład z-score maksymalnych kwot transakcji')
plt.xlabel('Z-score')
plt.ylabel('Liczba użytkowników')
plt.show()